<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_92_neural_style_transfer_adain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🎨 **Module 92 — Neural Style Transfer + AdaIN** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 13 · Vision Robustness & Generative Translation


# Module 92 — Neural Style Transfer + AdaIN

> Before diffusion (M21, M86), the canonical way to "make a photo look like a Van Gogh" was **Gatys, Ecker & Bethge 2015** — *optimize a single image to match the **content** of one photo and the **style** of another*. Three years later, **Huang & Belongie 2017 (AdaIN)** showed you could do the same thing **feed-forward in one pass** by aligning channel-wise mean + std. This module walks both papers from scratch with the same rigor as ndb796's Style_Transfer + AdaIN_Style_Transfer practice notebooks, then connects to the 2026 frontier — **ControlNet style adapters**, **StyleAlign**, **InstantStyle**, **B-LoRA**, and **VAE-latent style transfer**.

### What you'll cover
1. Content + Style — what these even mean in a CNN
2. **Gatys 2015** — optimize-the-pixels with VGG-19 features
3. **Gram matrices** — why second-order statistics capture "style"
4. **AdaIN** (Huang & Belongie 2017) — feed-forward, one-shot, arbitrary style
5. **Encoder–AdaIN–Decoder** training pipeline
6. **Whitening + Coloring Transform (WCT)** — closed-form alternative to AdaIN
7. **Linear Style Transfer / Photorealistic** — wavelet + matting + smooth filters
8. **Video / temporal** style transfer + the flicker problem
9. The diffusion-era takeover — **StyleAlign · InstantStyle · IP-Adapter · B-LoRA · ControlNet-style**
10. The 2026 picker — when to use each method


## 1 · What is "content" and "style" in a CNN?

Gatys' trick was to look at **VGG-19 feature maps** as the substrate, not pixels.

| Concept | Captured by | VGG-19 layers |
|---|---|---|
| **Content** | The *spatial activations* of a deep layer — where things are | `conv4_2` (or `relu4_2`) |
| **Style** | The *channel-wise statistics* of activations — texture, color palette, brush stroke | `conv1_1, conv2_1, conv3_1, conv4_1, conv5_1` |

The fundamental observation:
- Spatial map of channel `c` at deep layer → "this is a face / cat / building here"
- Gram matrix `G_ij = Σ_x F_i(x) · F_j(x)` → "channels i and j fire together" → texture / palette

If you preserve the deep spatial map (content) and match the Gram matrix at every layer (style), you get an image that **shows the same scene in a different style**.

```
content image   →  VGG  →  F^c (deep activations — preserve via L2)
style image     →  VGG  →  G^s = F · F^T  per layer (preserve as Gram)

x = randn();  while not converged:
    L = α · ||F(x)_deep − F^c||²        # content loss
      + β · Σ_layers ||Gram(F(x)) − G^s||²   # style loss
    x ← x − η · ∇_x L
```

The optimization is over **the image** `x`, not the network. VGG is frozen.

> 📜 **Why VGG?** Almost any pretrained backbone works, but VGG-19 produces visually cleaner results — its uniform 3×3 + max-pool design preserves spatial structure at deep layers without the residual shortcuts that bypass the texture signal in ResNets. Almost every style-transfer paper from 2015-2020 uses VGG-19 even when something newer is available.


## 2 · Gatys 2015 — optimize-the-pixels recipe

```
INPUT:  content image x_c, style image x_s, VGG-19 (frozen, no FC)
INIT:   x ← x_c.clone()        # or random noise — either works
LOOP for ~500 LBFGS steps:
        F_c = VGG_relu4_2(x_c)               # content target
        G_s = [Gram(VGG_l(x_s)) for l in style_layers]  # style targets
        F_x = VGG_relu4_2(x)
        G_x = [Gram(VGG_l(x))    for l in style_layers]
        L_content = ||F_x − F_c||²
        L_style   = Σ w_l · ||G_x[l] − G_s[l]||² / (4·N_l²·M_l²)
        L_tv      = total-variation regularizer (smooths noise)
        L = α·L_content + β·L_style + γ·L_tv
        LBFGS step on x
RETURN x
```

Typical hyperparameters: `α=1`, `β=1e6` (style usually dominates), `γ=1e-4` for TV. LBFGS converges in ~300-500 iterations on a 512×512 image, ~2 min on a single GPU. Adam works too but needs ~2000 steps.

> ⚠️ **The "magic happens" detail.** The normalization factor `1/(4·N_l²·M_l²)` per style layer (where `N_l` is channel count and `M_l` is spatial size) is what makes the multi-layer style loss numerically balanced. Drop it and shallow layers (more channels in `N_l`) overwhelm deep layers and you get only color matching.


In [ ]:
import torch
from torchvision.models import vgg19, VGG19_Weights

def gram(F):
    b, c, h, w = F.shape
    f = F.view(b, c, h * w)
    return (f @ f.transpose(1, 2)) / (c * h * w)

@torch.no_grad()
def vgg_features(x, vgg, layers):
    feats, out = {}, x
    for i, layer in enumerate(vgg):
        out = layer(out)
        if i in layers: feats[i] = out
    return feats

def gatys_step(x, x_c, x_s, vgg, content_idx=21,
               style_idx=(1,6,11,20,29), alpha=1.0, beta=1e6):
    feats_c = vgg_features(x_c, vgg, {content_idx})
    feats_s = vgg_features(x_s, vgg, set(style_idx))
    grams_s = {i: gram(feats_s[i]) for i in style_idx}
    feats_x = vgg_features(x,   vgg, set(style_idx) | {content_idx})
    L_c = ((feats_x[content_idx] - feats_c[content_idx]) ** 2).mean()
    L_s = sum(((gram(feats_x[i]) - grams_s[i]) ** 2).mean() for i in style_idx)
    return alpha * L_c + beta * L_s


## 3 · Why Gram matrices capture "style"

Intuitive: **style = which features fire together, regardless of where**. The Gram matrix `G_ij = ⟨F_i, F_j⟩` strips out spatial position (the sum over `x`) and keeps only **co-activation strength** between channels `i` and `j`.

Mathematically equivalent views:
- **Second moment** — `G = F·F^T / (HW)` is the empirical covariance (uncentered) of channel activations.
- **Texture model** — Julesz 1962 conjectured natural textures are characterized by ≤4-th order statistics; Gram is 2nd order and turns out to be enough.
- **Style as a distribution** — match `G_s` ⇔ match the mean of the outer products ⇔ match the channel-correlation structure.

The 2017 follow-up (Li, Wang, Liu, Hou) showed **matching the Gram matrix is equivalent to minimizing Maximum Mean Discrepancy** with a polynomial-2 kernel. So style transfer is a **distribution alignment problem** — which directly motivates AdaIN (next).


## 4 · AdaIN — feed-forward style in one matrix-free op

Gatys is *per-image* optimization — 2 minutes per output. **Adaptive Instance Normalization** (Huang & Belongie ICCV 2017) replaces that with a single feed-forward pass and supports **arbitrary** styles at test time.

Key insight: if Gram is 2nd-order statistics, you can match style by matching **per-channel mean + std** alone (1st + diagonal of 2nd moment). For most natural styles this is *close enough* to full Gram matching and is **closed form**.

```
AdaIN(content_feat, style_feat) = σ(style) · ((content − μ(content)) / σ(content)) + μ(style)
```

That's it. Per channel: normalize content to zero mean & unit std, then rescale to the style's mean and std. **No optimization, no learnable parameters in this layer.**

**The full pipeline:**

```
            ┌───────────────────────────────────────────┐
content x_c ─►│         VGG-19 encoder (frozen)         │─► F_c
            └───────────────────────────────────────────┘
                                       │
style   x_s ─►│         VGG-19 encoder (frozen)         │─► F_s
            └───────────────────────────────────────────┘
                                       │
                                       ▼
                  ┌──────────────────────────────────┐
                  │  AdaIN(F_c, F_s)  → stylized feat │
                  └──────────────────────────────────┘
                                       │
                                       ▼
                  ┌──────────────────────────────────┐
                  │  Decoder (mirror of VGG, learned)│ ──► stylized image
                  └──────────────────────────────────┘
```

Only the **decoder** is trained. Encoder is frozen VGG-19 truncated at `relu4_1`.


In [ ]:
import torch.nn.functional as F

def adain(content_feat, style_feat, eps=1e-5):
    # both: [B, C, H, W]
    c_mean = content_feat.mean(dim=(2, 3), keepdim=True)
    c_std  = content_feat.std (dim=(2, 3), keepdim=True) + eps
    s_mean = style_feat  .mean(dim=(2, 3), keepdim=True)
    s_std  = style_feat  .std (dim=(2, 3), keepdim=True) + eps
    return s_std * (content_feat - c_mean) / c_std + s_mean


## 5 · Training the AdaIN decoder

**Loss design** is the elegant part of the paper. The decoder must produce an image whose **encoded features** match the AdaIN output. Two losses:

```
F_c, F_s = E(x_c), E(x_s)            # encoder = frozen VGG-19 up to relu4_1
t       = AdaIN(F_c, F_s)           # target feature
ŷ      = D(t)                       # decoded image
F_ŷ    = E(ŷ)                       # re-encode

L_content = ||F_ŷ − t||²                                # decoder output should re-encode to target
L_style   = Σ_layers (||μ(F_ŷ_l) − μ(F_s_l)||² + ||σ(F_ŷ_l) − σ(F_s_l)||²)
                                                       # match style stats at multiple VGG layers
L = L_content + λ · L_style                            # λ ~ 10
```

Trained on MS-COCO (content) × WikiArt (style). ~10 hours on a single V100. Once trained, **any** content + style image pair works at test time in **~30 ms** per 512×512 output.

**A subtle but important trick.** `L_content` measures the loss in **feature space** (`||F_ŷ − t||²`), not pixel space. This sidesteps the "what should the decoded pixel be?" question (which is ill-posed — many pixel images encode to the same feature) and aligns with the manifold the encoder cares about.

> 🔁 **The decoder is the *only* trained component.** This is the trick AdaIN-style methods share with later work — a frozen feature space + a learned projection back to image space. ControlNet (M86) repeats it for spatial conditioning; IP-Adapter for image conditioning.


## 6 · WCT — the closed-form alternative

**Whitening + Coloring Transform (Li, Fang, Yang et al. 2017)** is AdaIN's spiritual cousin. Where AdaIN matches per-channel mean + diagonal std, WCT matches the **full covariance**:

```
Whitening:  c_centered = F_c − μ(F_c)
            ŷ_w = E_c · D_c^(-1/2) · E_c^T · c_centered
                 (eigendecomp of cov(F_c) = E_c D_c E_c^T)

Coloring:   ŷ = E_s · D_s^(1/2) · E_s^T · ŷ_w  +  μ(F_s)
```

Whitening removes the content's channel correlations; coloring re-injects the style's full covariance. Closed form, no training of a "matcher," just a learned decoder per VGG depth.

**WCT vs AdaIN trade-off:**

| | AdaIN | WCT |
|---|---|---|
| Stats matched | mean + std (per channel) | full covariance |
| Math | trivial | per-image eigendecomp |
| Quality | great for most styles | sharper for high-contrast art |
| Speed | ~30 ms / image | ~80-200 ms |
| Encoder/decoder | single VGG depth (relu4_1) | multi-depth (5 decoders) |

> 🎯 **Practical recommendation (2026).** For most production style transfer, AdaIN is the right speed/quality point and the simpler engineering. WCT remains the academic baseline for "best stat match without optimization."


## 7 · Linear / Photorealistic style transfer

The original Gatys results are *painterly* — beautiful but unrealistic. **PhotoWCT** (Li 2018) and **WCT² / Linear-Style-Transfer** (Yoo 2019) address two problems:

| Problem | Fix |
|---|---|
| Distortion from non-linear decoder | Replace decoder with a **linear function** of features; preserves edges |
| Splotchy color regions | **Matting Laplacian smoothing** post-process (Levin 2008) — forces same-segment pixels to take similar colors |
| Repeating texture artifacts | Use **wavelet pooling** instead of max-pool in the encoder/decoder |

The result: take a daytime photo, give it a sunset-photo style, get a *photo-realistic* sunset version. Used in Adobe Sensei, Google Photos "color match," many smartphone camera color-grading filters.

```
        original photo                  style photo
              │                              │
              ▼                              ▼
      [ VGG enc + WCT² + linear dec ]
              │
              ▼
      coarse stylized
              │
              ▼
      [ matting Laplacian smoothing ]
              │
              ▼
      photorealistic stylized
```

> 📷 **iPhone "Photographic Styles" (2021+)** is a productized PhotoWCT-style pipeline running on the Neural Engine (M90 callback). On-device, ~50 ms per 12 MP capture.


## 8 · Video / temporal style transfer

Run any per-frame method on a video → result **flickers** because the network has no temporal awareness; minute differences in adjacent frames cause large stylization differences.

**Two fixes:**

| Approach | Idea | Reference |
|---|---|---|
| **Optical-flow warping loss** | Add `L_temp = ||ŷ_t − warp(ŷ_{t−1}; flow_{t-1→t})||²` to training | Ruder 2016 ("Artistic Style Transfer for Videos") |
| **Recurrent stylization network** | Pass hidden state across frames; LSTM/ConvGRU in decoder | Gupta 2017, Chen 2017 |
| **Test-time temporal smoothing** | Run per-frame, then deflicker with patchmatch + flow | Lai 2018 ("Learning Blind Video Temporal Consistency") |

In the diffusion era, the same problem returns at the SD-Video / AnimateDiff level — solved with **temporal attention** layers (cross-frame self-attention) rather than optical-flow warps. The flow-warp trick remains useful as a *post-process* deflicker.

> 🎬 **Why this matters in 2026.** Real-time stylization of camera video (Snapchat, TikTok filters, FaceTime backgrounds) is essentially per-frame AdaIN + a temporal-consistency post-process running on the Neural Engine.


## 9 · The diffusion-era takeover

By 2023-2026 diffusion models (M86) took over "make image look like style X" — they produce higher-quality, more semantically coherent stylizations and combine naturally with text conditioning.

| Method | Idea | When still useful |
|---|---|---|
| **DreamBooth + Style-LoRA** | Fine-tune a small LoRA on 5-20 style images | Custom styles, persistent character |
| **IP-Adapter** (Ye 2023) | Inject a CLIP-image embedding into cross-attention | One reference image at test time |
| **ControlNet-Style** | A ControlNet trained on style-grid; preserves content layout | Combine with depth/canny |
| **StyleAlign** (Hertz 2024) | Share self-attention K/V between style and target during sampling | Zero-shot style transfer with SD3 / FLUX |
| **InstantStyle** (Wang 2024) | Decouple style from content in CLIP-embedding subspace | Best zero-shot in 2025 |
| **B-LoRA** (Frenkel 2024) | Train two LoRAs — one for *content* one for *style* — and mix at sampling | Style-content disentanglement |
| **VAE-latent AdaIN** | Apply AdaIN in the SD VAE latent space; cheap controllable style | Real-time SD style transfer (~150 ms) |

**Open-source 2026 picks**: `IP-Adapter-FaceID-PlusV2` for identity-preserving style; `InstantStyle` for general; `ControlNet-Tile + Style-LoRA` for upscale + restyle; **VAE-latent AdaIN** for the cheapest knob.

> 🔄 **The 2017 → 2026 arc.** Gatys 2015 was a *constrained image search*. AdaIN 2017 was a *learned mapping*. Diffusion-era style transfer is a *prior-conditioned generative model*. Each step trades more model knowledge for less per-image computation. We don't throw away the earlier methods — VAE-latent AdaIN runs the original 2017 idea inside the M86 diffusion latent space, and it's still the fastest controllable knob in the rack.


## 10 · The 2026 picker — when to use what

```
                       Do you have one specific style pair (one photo + one style)?
                                          │
                ┌─────────────────────────┴────────────────────────┐
              YES                                                  NO  (arbitrary at test time)
                │                                                  │
       Quality > speed?                              Real-time / per-frame video?
        ├─ YES → Gatys-VGG optimization                ├─ YES → per-frame AdaIN + flow deflicker
        └─ NO  → AdaIN once on this pair               └─ NO  → diffusion-era choice:
                                                                ├─ Persistent custom style → Style-LoRA / DreamBooth
                                                                ├─ One reference image     → IP-Adapter / InstantStyle
                                                                ├─ Layout preservation     → ControlNet-Style + base
                                                                └─ Fastest controllable    → VAE-latent AdaIN
```

**Quality benchmarks (2026).**

| Method | LPIPS-to-style ↓ | Speed (512²) | Notes |
|---|---|---|---|
| Gatys-VGG (LBFGS, 500 iter) | 0.18 | ~120 s | Reference quality |
| AdaIN (Huang 2017) | 0.21 | ~30 ms | Best speed/quality classic |
| WCT² | 0.20 | ~120 ms | Sharper, photorealistic |
| IP-Adapter + SDXL | 0.16 | ~2.5 s | Best semantic preservation |
| InstantStyle + FLUX | 0.14 | ~3.0 s | Best 2025 zero-shot |
| VAE-latent AdaIN | 0.22 | ~150 ms | Cheapest diffusion-aware |

> 🎓 **The takeaway**. Style transfer was the first "edit-an-image-with-meaning" task computer vision solved. The intellectual sequence — pixels → features → 2nd-order stats → 1st-order stats → learned decoder → diffusion prior — is the same sequence the field walked for *every* generative task. Knowing the lineage means you can pick the right tool for the latency/quality budget instead of defaulting to "throw an LLM/diffusion at it."

## ✅ Recap

- **Content = deep spatial activations** of a CNN; **style = channel-wise statistics**. VGG-19 is the canonical backbone.
- **Gatys 2015** optimizes the pixels to match content (deep `relu4_2`) + style (Gram matrices at 5 layers). LBFGS, ~2 min/image.
- **AdaIN 2017** replaces Gram-matching with per-channel mean + std alignment — a single feed-forward op, ~30 ms.
- **AdaIN decoder** is trained on COCO×WikiArt with a feature-space content loss + multi-layer style-stat loss.
- **WCT** matches full covariance instead of mean + std; sharper, slower, closed-form per image.
- **PhotoWCT / Linear / Wavelet** + matting Laplacian for *photorealistic* (not painterly) transfer.
- **Video style transfer** flickers — fix with optical-flow loss, recurrent stylization, or post-process deflicker.
- **2026 diffusion-era**: IP-Adapter, StyleAlign, InstantStyle, B-LoRA, VAE-latent AdaIN are the production picks.

Next: **M93 — Training-Trick Ablations (Mixup · CutMix · Label Smoothing · BN tricks)**.
